# Module 2 - Partie 4 : Découpage et pipeline scikit-learn

## Objectif

Mettre en place un premier protocole de modélisation pour détecter les fraudes du dataset PaySim, qu'on puisse relancer à l'identique.

Le notebook découpe les données dans l'ordre chronologique, garde six variables connues au moment de la transaction, les prétraite dans des pipelines scikit-learn, puis compare un modèle naïf à une régression logistique. Les modèles sont ajustés sur l'entraînement et comparés sur la validation. Le jeu de test n'est pas utilisé ici.

L'évaluation détaillée et le choix du modèle sont traités en partie 5.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


In [2]:
fichier_parquet = Path("../paysim_echantillon.parquet")

print("Fichier trouvé :", fichier_parquet.exists())

df = pd.read_parquet(fichier_parquet)

print("Dimensions :", df.shape)
display(df.head())


Fichier trouvé : True


Dimensions : (555108, 10)


,tx_id,step,heure,type,amount,oldbalanceOrg,oldbalanceDest,nb_tx_orig_avant,nb_recu_dest_avant,isFraud
0,48,1,0,CASH_OUT,56953.90,1942.02,70253.0,0,0,0
1,59,1,0,TRANSFER,62610.80,79114.00,517.0,0,0,0
2,73,1,0,CASH_OUT,94253.33,25203.05,99773.0,0,0,0
3,84,1,0,TRANSFER,125872.53,0.00,348512.0,0,0,0
4,86,1,0,TRANSFER,1505626.01,0.00,29031.0,0,0,0


## 1. Découpage chronologique : méthode et justification

En situation réelle, un modèle est entraîné sur des transactions passées et appliqué aux suivantes. Le découpage suit donc l'ordre de `step`. Un tirage aléatoire mélangerait passé et futur dans chaque ensemble, ce qui cacherait l'évolution de la fraude dans le temps et risquerait de donner des scores trop optimistes.

Les frontières sont les quantiles 60 % et 80 % de `step`. Le quantile est calculé sur les transactions (une ligne par transaction), pas sur les steps : les steps chargés pèsent plus, et la durée simulée n'est pas coupée en trois parts égales.

Sur cet échantillon : entraînement, steps 1 à 281 ; validation, steps 282 à 354 ; test, steps 355 à 743. Le test couvre plus de la moitié des steps pour environ 20 % des transactions.

Le taux de fraude change beaucoup d'une période à l'autre : 0,18 % en entraînement, 0,12 % en validation, 0,76 % en test (calcul plus bas). La distribution n'est donc pas stable dans le temps. Les résultats de validation se lisent avec cette réserve, et le test reste réservé à l'évaluation finale.

In [3]:
print("Période couverte (step) :", df["step"].min(), "à", df["step"].max())

print()
print("Répartition de la variable cible :")
print(df["isFraud"].value_counts())

print()
print("Taux de fraude (%) :")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))

print()
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

Période couverte (step) : 1 à 743

Répartition de la variable cible :
isFraud
0    553512
1      1596
Name: count, dtype: int64

Taux de fraude (%) :
isFraud
0    99.7125
1     0.2875
Name: proportion, dtype: float64

Valeurs manquantes par colonne :
tx_id                 0
step                  0
heure                 0
type                  0
amount                0
oldbalanceOrg         0
oldbalanceDest        0
nb_tx_orig_avant      0
nb_recu_dest_avant    0
isFraud               0
dtype: int64


In [4]:
# Les quantiles de step sont calculés sur toutes les transactions.
# Chaque transaction contribue donc une observation au calcul du quantile.
borne_train = int(df["step"].quantile(0.60))
borne_validation = int(df["step"].quantile(0.80))

print("Fin entraînement :", borne_train)
print("Fin validation :", borne_validation)
print("Début test :", borne_validation + 1)

# Affectation des périodes pour contrôler les effectifs et les taux de fraude.
df_verification = df.copy()
df_verification["periode"] = pd.Series(index=df_verification.index, dtype="object")
df_verification.loc[df_verification["step"] <= borne_train, "periode"] = "Entrainement"
df_verification.loc[
    (df_verification["step"] > borne_train)
    & (df_verification["step"] <= borne_validation),
    "periode",
] = "Validation"
df_verification.loc[df_verification["step"] > borne_validation, "periode"] = "Test"


Fin entraînement : 281
Fin validation : 354
Début test : 355


In [5]:
resume = df_verification.groupby("periode").agg(
    transactions=("isFraud", "size"),
    fraudes=("isFraud", "sum"),
)

resume["part_transactions_pct"] = (
    100 * resume["transactions"] / len(df)
).round(2)
resume["taux_fraude_pct"] = (
    100 * resume["fraudes"] / resume["transactions"]
).round(4)

print(resume)


              transactions  fraudes  part_transactions_pct  taux_fraude_pct
periode                                                                    
Entrainement        335288      617                  60.40           0.1840
Test                110516      843                  19.91           0.7628
Validation          109304      136                  19.69           0.1244


In [6]:
# Création des trois ensembles chronologiques
df_train = df[df["step"] <= borne_train].copy()
df_val = df[
    (df["step"] > borne_train)
    & (df["step"] <= borne_validation)
].copy()
df_test = df[df["step"] > borne_validation].copy()

print("Entraînement :", df_train.shape)
print("Validation   :", df_val.shape)
print("Test         :", df_test.shape)

# Vérification de l'ordre, de l'absence de chevauchement et de l'exhaustivité.
assert df_train["step"].max() < df_val["step"].min()
assert df_val["step"].max() < df_test["step"].min()
assert len(df_train) + len(df_val) + len(df_test) == len(df)
print("Découpage chronologique vérifié.")


Entraînement : (335288, 10)
Validation   : (109304, 10)
Test         : (110516, 10)
Découpage chronologique vérifié.


## 2. Variables prédictives et prévention des fuites

La cible est `isFraud` (0 pour une transaction légitime, 1 pour une fraude). Six variables servent à la prédire : `type`, `amount`, `oldbalanceOrg`, `oldbalanceDest`, `nb_tx_orig_avant` et `nb_recu_dest_avant`.

`step` sert seulement à découper les périodes. `tx_id` est un identifiant technique. `heure` est dérivée de `step` et n'entre pas dans ce premier modèle. `newbalanceOrig` et `newbalanceDest` ne sont connues qu'après la transaction, donc elles sont écartées, de même que `isFlaggedFraud`.

In [7]:
# Variables explicatives retenues
variables = [
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "nb_tx_orig_avant",
    "nb_recu_dest_avant"
]

cible = "isFraud"

# Séparation des variables explicatives et de la cible
X_train = df_train[variables].copy()
y_train = df_train[cible].copy()

X_val = df_val[variables].copy()
y_val = df_val[cible].copy()

X_test = df_test[variables].copy()
y_test = df_test[cible].copy()

print("Variables retenues :", X_train.columns.tolist())
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)

Variables retenues : ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'nb_tx_orig_avant', 'nb_recu_dest_avant']
X_train : (335288, 6)
y_train : (335288,)
X_val   : (109304, 6)
X_test  : (110516, 6)


## 3. Prétraitement scikit-learn

Un `ColumnTransformer` traite les colonnes en deux branches. Les cinq variables numériques passent par une imputation par la médiane puis un `StandardScaler`. `type` passe par une imputation par la modalité la plus fréquente puis un `OneHotEncoder`.

Le préprocesseur défini ci-dessous n'est pas ajusté : c'est une configuration, et chaque pipeline en reçoit une copie avec `clone`. Quand on appelle `Pipeline.fit(X_train, y_train)`, les valeurs d'imputation, l'échelle et les catégories sont apprises sur l'entraînement seul. La validation est ensuite transformée avec ces paramètres, sans nouvel ajustement.

In [8]:
variables_numeriques = [
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "nb_tx_orig_avant",
    "nb_recu_dest_avant",
]
variables_categorielles = ["type"]

pipeline_numerique = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="median")),
    ("standardisation", StandardScaler()),
])
pipeline_categoriel = Pipeline(steps=[
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore")),
])

preprocesseur = ColumnTransformer(transformers=[
    ("numerique", pipeline_numerique, variables_numeriques),
    ("categoriel", pipeline_categoriel, variables_categorielles),
])
print("Préprocesseur non ajusté créé.")


Préprocesseur non ajusté créé.


## 4. Modèle naïf de référence

Avant la régression logistique, un `DummyClassifier` sert de point de comparaison.

Avec la stratégie `"prior"`, il prédit toujours la classe majoritaire, et ses probabilités sont les fréquences des classes dans l'entraînement.

Il montre pourquoi l'accuracy ne suffit pas quand les classes sont aussi déséquilibrées : il ne signale aucune fraude et a pourtant raison sur plus de 99 % des transactions.

In [9]:
# Copie indépendante du préprocesseur pour ce pipeline.
pipeline_naif = Pipeline(steps=[
    ("pretraitement", clone(preprocesseur)),
    ("modele", DummyClassifier(strategy="prior")),
])

# Ajustement sur les seules données d'entraînement.
pipeline_naif.fit(X_train, y_train)

print("Modèle naïf entraîné sur l'entraînement.")
print("Classes :", pipeline_naif.named_steps["modele"].classes_)
print("Fréquences des classes :", pipeline_naif.named_steps["modele"].class_prior_)


Modèle naïf entraîné sur l'entraînement.
Classes : [0 1]
Fréquences des classes : [0.99815979 0.00184021]


## 5. Régression logistique pondérée

La régression logistique est le premier modèle supervisé. L'option `class_weight="balanced"` donne plus de poids aux fraudes, avec des poids calculés à partir des fréquences de l'entraînement. Le pipeline a sa propre copie du préprocesseur et n'est ajusté que sur l'entraînement.

In [10]:
pipeline_logistique = Pipeline(steps=[
    ("pretraitement", clone(preprocesseur)),
    ("modele", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

pipeline_logistique.fit(X_train, y_train)

print("Régression logistique entraînée sur l'entraînement.")
print("Nombre de variables après prétraitement :",
      pipeline_logistique.named_steps["modele"].n_features_in_)


Régression logistique entraînée sur l'entraînement.
Nombre de variables après prétraitement : 7


## 6. Comparaison des modèles sur la validation

Les prédictions et les métriques portent sur la validation uniquement. Le seuil de décision reste à 0,5 pour les deux modèles, sans optimisation à ce stade.

In [11]:
# Prédictions sur la validation uniquement ; aucun appel au jeu de test.
pred_naif = pipeline_naif.predict(X_val)
pred_logistique = pipeline_logistique.predict(X_val)

print("Transactions en validation :", len(X_val))
print("Transactions signalées par le modèle naïf :", int((pred_naif == 1).sum()))
print("Transactions signalées par la régression logistique :",
      int((pred_logistique == 1).sum()))


Transactions en validation : 109304
Transactions signalées par le modèle naïf : 0
Transactions signalées par la régression logistique : 8718


In [12]:
# Les métriques sont recalculées depuis les prédictions, sans valeurs codées en dur.
tableau_comparaison = pd.DataFrame([
    {
        "Modèle": "Modèle naïf",
        "Transactions signalées": int((pred_naif == 1).sum()),
        "Précision": precision_score(y_val, pred_naif, zero_division=0),
        "Rappel": recall_score(y_val, pred_naif, zero_division=0),
        "F1-score": f1_score(y_val, pred_naif, zero_division=0),
    },
    {
        "Modèle": "Régression logistique",
        "Transactions signalées": int((pred_logistique == 1).sum()),
        "Précision": precision_score(y_val, pred_logistique, zero_division=0),
        "Rappel": recall_score(y_val, pred_logistique, zero_division=0),
        "F1-score": f1_score(y_val, pred_logistique, zero_division=0),
    },
]).set_index("Modèle")

# Affichage du tableau sans dépendance supplémentaire
tableau_affichage = tableau_comparaison.copy()

for colonne in ["Précision", "Rappel", "F1-score"]:
    tableau_affichage[colonne] = tableau_affichage[colonne].map(
        lambda valeur: f"{valeur:.2%}"
    )

display(tableau_affichage)


,Transactions signalées,Précision,Rappel,F1-score
Modèle,,,,
Modèle naïf,0,0.00%,0.00%,0.00%
Régression logistique,8718,1.31%,83.82%,2.58%


## 7. Sauvegarde du pipeline de référence

Le pipeline de régression logistique ajusté sur l'entraînement est enregistré avec `joblib.dump` dans `pipeline_reference.joblib`. Le fichier contient le prétraitement et le modèle : on peut le recharger et l'appliquer à des transactions brutes qui ont les six colonnes de la section 2, sans rien réajuster.

Un fichier joblib se recharge avec la version de scikit-learn qui l'a produit, affichée ci-dessous. Le seuil de décision n'est pas dans le fichier : il est choisi en partie 5.

In [13]:
import joblib
import sklearn

fichier_pipeline = Path("pipeline_reference.joblib")
joblib.dump(pipeline_logistique, fichier_pipeline)

print("Pipeline sauvegardé :", fichier_pipeline, f"({fichier_pipeline.stat().st_size / 1024:.1f} Ko)")
print("Version de scikit-learn :", sklearn.__version__)

# Le fichier rechargé doit donner les mêmes scores que le pipeline en mémoire.
pipeline_recharge = joblib.load(fichier_pipeline)
assert (pipeline_recharge.predict_proba(X_val) == pipeline_logistique.predict_proba(X_val)).all()
print("Pipeline rechargé : scores identiques sur la validation.")

Pipeline sauvegardé : pipeline_reference.joblib (4.3 Ko)
Version de scikit-learn : 1.7.2


Pipeline rechargé : scores identiques sur la validation.


## 8. Conclusion et périmètre de la partie 5

Au seuil de 0,5, la régression logistique signale 8 718 transactions sur les 109 304 de la validation. Elle retrouve 83,82 % des 136 fraudes, mais sa précision est de 1,31 % : presque toutes les alertes sont de fausses alertes. Le F1-score est de 2,58 %. Le modèle naïf ne signale rien, donc ses trois métriques sont à 0.

Ces chiffres sont une première comparaison, pas une mesure de généralisation. Le taux de fraude varie d'une période à l'autre, et le jeu de test n'a servi ni à ajuster les modèles ni à calculer ces résultats.

Le pipeline de régression logistique est sauvegardé dans `pipeline_reference.joblib`.

Restent pour la partie 5 : la validation croisée temporelle, l'évaluation finale sur le test, des métriques adaptées au déséquilibre, des intervalles de confiance et le choix du seuil de décision.